# Chatbot with streaming

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

In [3]:
from sarvamai import SarvamAI

client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Use Case Definition

Build a chatbot for ABC Bank

The chatbot interacts with customers online helping them:
1. Check the Account Balance
2. Review the list of transactions in their account in the last 24 hours

## Setup the tools

In [4]:
def get_balance(account_number: str):
    if account_number == '001002':
        return "INR 51203"
    else:
        return "INR 2500"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
     {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get transactions from the last 24 hours",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },   
]

In [6]:
tools_map  = {
    "get_balance": get_balance,
    "get_transactions": get_transactions,
}

def invoke_tool(f_name: str, f_args: dict):
    return tools_map[f_name](**f_args)

### Code the chatbot

In [7]:
import json

def handle_tool_calls(tool_calls: list, messages: list):    
    for tool_call in tool_calls:
        f_name = tool_call["function"]["name"] #get_balance
        f_args = json.loads(tool_call["function"]["arguments"]) # {"account_number": "001002"}
        result = invoke_tool(f_name, f_args)

        messages.append(
            {
                "role": "assistant",
                "tool_calls": [
                    {
                        "id": tool_call["id"],
                        "type": "function",
                        "function": {
                            "name": f_name,
                            "arguments": tool_call["function"]["arguments"],
                        },
                    }
                ],
            }
        )
            
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call["id"],
                "content": str(result),
            }
        )   

In [8]:
BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

def run_chatbot():

    messages=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ]
    
    while True:
        user_message = input("\nUser:")
        
        if user_message == "quit":
            break
            
        messages.append(
            {"role": "user", "content": user_message.strip()}
        )

        while True:
            stream = client.chat.completions(
                model="sarvam-105b-conversations",
                messages=messages,
                tools=tools,
                stream=True,
            )
    
            tool_calls = []
            assistant_message = ""
            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    assistant_message += chunk.choices[0].delta.content
                    break
                if chunk.choices and chunk.choices[0].delta.tool_calls:
                    current_tool_call_chunk = chunk.choices[0].delta.tool_calls[0]
                    idx = current_tool_call_chunk.index
                    if idx == len(tool_calls):
                        tool_calls.append(
                            {
                                "index": idx,
                                "id": "",
                                "type": "function",
                                "function": {
                                    "name": "",
                                    "arguments": "",
                                }
                            }
                        )
                    current_tool_call = tool_calls[-1]
                    if id_ := current_tool_call_chunk.id:
                        current_tool_call["id"] += id_
                    if function_ := current_tool_call_chunk.function:
                        if name_ := function_.name:
                            current_tool_call["function"]["name"] += name_
                        if arguments_ := function_.arguments:
                            current_tool_call["function"]["arguments"] += arguments_

            if tool_calls:
                handle_tool_calls(tool_calls, messages)
                continue
                            
            print("\n", BLUE_TEXT, "Assistant: ", assistant_message, end="",flush=True)
            for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    assistant_message += chunk.choices[0].delta.content
                    print(chunk.choices[0].delta.content, end="",flush=True)
            print(BLACK_TEXT, end="", flush=True)
            messages.append(
                {"role": "assistant", "content": assistant_message}
            )
            break
                

            

In [ ]:
run_chatbot()


User: hi



  Assistant:  Hello! Welcome to ABC Bank. How can I assist you today?


User: How are you ?



  Assistant:  I'm doing well, thank you for asking! I'm here to help you with any banking needs you may have. Is there something specific I can assist you with today?


User: please get my account balance and transaction details


...


  Assistant:  Here are your account details for account number 001002:

**Account Balance:** INR 51,203

**Recent Transactions (Last 24 Hours):**
- 11:21 AM — Debit — XYZ Super Market — INR 651.50
- 4:45 PM — Credit — Refund from PQR Broadband Services — INR 999.00

Is there anything else you'd like to know or any other way I can assist you today?


User: what is sarvam AI ? provide the details after displaying balance and transactions for my account



  Assistant:  Here are your account details for account number 001002:

**Account Balance:** INR 51,203

**Recent Transactions (Last 24 Hours):**
- 11:21 AM — Debit — XYZ Super Market — INR 651.50
- 4:45 PM — Credit — Refund from PQR Broadband Services — INR 999.00

Regarding your question about Sarvam AI — I'm actually an AI assistant created by **Sarvam AI**, which is a technology company that develops artificial intelligence solutions. As a customer service representative for ABC Bank, I'm here to help you with your banking needs. Sarvam AI focuses on building AI models and tools, including language and customer service applications like me.

If you have any other questions about your account or need further assistance, please let me know!


User: provide a detailed version of Sarvam Ai 



  Assistant:  Here is a more detailed overview of **Sarvam AI**:

**Company Overview**
Sarvam AI is an artificial intelligence research and development company focused on building advanced AI models and solutions. They specialize in natural language processing, machine learning, and AI-driven applications designed for Indian languages and contexts, though their technologies serve global markets as well.

**Core Focus Areas**
- **Language Models:** Developing AI systems that understand and generate text in multiple languages, with strong capabilities in Indian languages.
- **Voice & Speech Technologies:** Creating solutions for speech recognition, text-to-speech, and conversational AI.
- **Customer Service & Enterprise AI:** Building intelligent assistants and automation tools for businesses, like the one you're interacting with today.
- **Research & Innovation:** Advancing the state of AI through ongoing research in areas like multilingual understanding and efficient model training.

